# Day 13/60 — Rate Limiter (Part 3): Deep Dive & Distributed Design

## System Design Series

**Topic:** The high-level design from Part 1 left several questions unanswered. This day answers them with a complete deep dive:

- Where to store rate-limit counters (database vs Redis)
- How rate limiting rules are created and stored
- How to handle rate-limited requests (drop vs enqueue)
- The full detailed request flow through middleware
- Race conditions and synchronisation challenges in distributed environments
- Complete architecture diagram tying all components together

> All flow diagrams in this notebook use **Mermaid** — rendered live in Jupyter.

---

## 1. Setup: Mermaid Renderer

In [1]:
from IPython.display import HTML

def mermaid(code):
    return HTML(
        '<div class="mermaid" style="max-width:960px;background:white;'
        'padding:24px;border-radius:10px;border:1px solid #ddd;margin:12px 0;">'
        + code +
        '</div>'
        '<script type="module">'
        'import md from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";'
        'md.initialize({startOnLoad:true,theme:"default",'
        'flowchart:{curve:"linear",padding:20},'
        'sequence:{useMaxWidth:true,mirrorActors:false}});'
        '</script>'
    )

print("Day 13 ready — Mermaid renderer loaded.")


Day 13 ready — Mermaid renderer loaded.


### What this does

Defines a `mermaid()` helper that renders Mermaid diagram source as live, interactive SVG inside Jupyter via CDN. No Matplotlib needed — each diagram is declared as a plain text flowchart or sequence diagram, rendered by the browser at display time.

**Mermaid diagram types used in this notebook:**
- `flowchart LR / TD` — left-to-right or top-down flow diagrams
- `sequenceDiagram` — actor-based message sequence diagrams
- `subgraph` — grouping related nodes into labelled containers

## 2. Where to Store Rate-Limit Counters

> "Using the database is not a good idea due to slowness of disk access. In-memory cache is chosen because it is fast and supports time-based expiration. Redis is the best option — it offers two commands: INCR and EXPIRE." — Day 13

**The two Redis commands that power rate limiting:**

| Command | Purpose |
|---|---|
| `INCR key` | Atomically increment the counter by 1; creates key if absent |
| `EXPIRE key ttl` | Set the key to auto-delete after `ttl` seconds — automatic window reset |

A SQL or NoSQL database requires a disk read and write per request (~1–10ms). At 10,000 req/s that adds 10–100 seconds of cumulative latency per second — clearly infeasible. Redis in-memory lookup takes ~0.1ms.

In [2]:
# Cell 1 — Where to store rate-limit counters (DB vs Redis)
display(mermaid("""
flowchart LR
    A([Client Request]) --> B{Store counter\nwhere?}

    B --> C[(SQL / NoSQL\nDatabase)]
    B --> D[(Redis\nIn-Memory)]

    C --> E[Disk read per\nrequest ~1-10 ms]
    E --> F([❌ Too slow\nfor rate limiting\non hot path])

    D --> G[In-memory\nread ~0.1 ms]
    G --> H[INCR — atomic\ncounter increment]
    G --> I[EXPIRE — auto\nreset after window]
    H --> J([✅ Chosen:\nFast + TTL support])
    I --> J

    style A fill:#2E86AB,color:#fff,stroke:none
    style B fill:#F39C12,color:#fff,stroke:none
    style C fill:#C44E52,color:#fff,stroke:none
    style D fill:#27AE60,color:#fff,stroke:none
    style E fill:#FDECEA,color:#C44E52,stroke:#C44E52
    style F fill:#C44E52,color:#fff,stroke:none
    style G fill:#EAFAF1,color:#27AE60,stroke:#27AE60
    style H fill:#2980B9,color:#fff,stroke:none
    style I fill:#2980B9,color:#fff,stroke:none
    style J fill:#27AE60,color:#fff,stroke:none
"""))


### What this shows

A decision flowchart forking at 'Store counter where?'. The database branch follows: disk read → too slow → ❌ red terminal. The Redis branch follows: in-memory read → INCR → EXPIRE → ✅ green terminal. The diagram makes the latency reason explicit rather than just stating 'use Redis'.

## 3. Rate Limiting Rules: Creation and Storage

> "Rate limiting rules are generally written in configuration format and saved on disk. Workers frequently pull rules from the disk and store them in the cache." — Day 13

**Rule examples (Lyft Envoy style — used in production):**

```yaml
domain: messaging
descriptors:
  - key: message_type
    value: marketing
rate_limit:
  unit: day
  requests_per_unit: 5
```

Rules are kept in config files (not the database) so they can be:
- Version-controlled alongside infrastructure code
- Deployed without a database migration
- Rolled back instantly if a rule causes false positives

The worker process pulls fresh rules on a configurable interval (typically 10–60 seconds) and populates an in-memory rules cache.

In [3]:
# Cell 2 — Rate limiting rules: where they live and how they flow
display(mermaid("""
flowchart TD
    subgraph Rules["Rate Limiting Rules Store"]
        direction TB
        RD[("📁 Rules on Disk\n(config files / YAML)")]
        W["⚙ Worker Process\n(pulls rules periodically)"]
        RC[("⚡ Rules Cache\n(Redis / in-memory)")]
        RD -->|"pull every\nN seconds"| W
        W -->|"write to\ncache"| RC
    end

    subgraph Flow["Request Flow"]
        direction TB
        CL([Client])
        RL["🛡 Rate Limiter\nMiddleware"]
        API["🖥 API Server"]

        CL -->|"HTTP request"| RL
        RL -->|"1 · load rules\nfrom cache"| RC
        RL -->|"2 · fetch counter\n+ last timestamp"| Redis
        Redis -->|"INCR / EXPIRE\nresult"| RL
    end

    Redis[("🔴 Redis\nCounter Store")]

    RL -->|"3a · under limit →\nforward"| API
    RL -->|"3b · over limit →\nHTTP 429"| CL

    style RD fill:#8E44AD,color:#fff,stroke:none
    style W fill:#E67E22,color:#fff,stroke:none
    style RC fill:#2980B9,color:#fff,stroke:none
    style CL fill:#2E86AB,color:#fff,stroke:none
    style RL fill:#E67E22,color:#fff,stroke:none
    style API fill:#8E44AD,color:#fff,stroke:none
    style Redis fill:#C44E52,color:#fff,stroke:none
"""))


### What this shows

Two subgraphs connected by arrows:

**Rules Store subgraph (top):** Rules on Disk → Worker (periodic pull) → Rules Cache. The worker is a background process that bridges the durable config store and the fast in-memory cache.

**Request Flow subgraph (bottom):** Client → Rate Limiter Middleware → Rules Cache (load rules) + Redis Counter Store (INCR/EXPIRE). Based on the result: forward to API server or return HTTP 429 to client. The separation of the rules cache from the counter store is deliberate — rules change slowly, counters change on every request.

## 4. Rate Limiting Rules — Examples

Three representative rule types cover the full range of rate limiting use cases:

| Rule type | Example | Purpose |
|---|---|---|
| Per-user per-type | 5 marketing messages / day / user | Spam prevention |
| Per-endpoint global | 5 login attempts / second | Brute-force protection |
| System-wide ceiling | 10,000 requests / second total | Capacity protection |

Rules can be composed — a request may match multiple rules, and the most restrictive rule that applies wins.

In [4]:
# Cell 3 — Rate limiting rules examples (Lyft Envoy config style)
display(mermaid("""
flowchart LR
    subgraph RuleSet["Rules Engine"]
        R1["Rule 1\n─────────────\ndomain: messaging\ndescriptors:\n  - key: message_type\n    value: marketing\nratelimit:\n  unit: day\n  requests_per_unit: 5"]

        R2["Rule 2\n─────────────\ndomain: auth\ndescriptors:\n  - key: auth.login\n    value: disable\nratelimit:\n  unit: second\n  requests_per_unit: 5"]

        R3["Rule 3\n─────────────\ndomain: global\ndescriptors:\n  - key: ip_address\nratelimit:\n  unit: second\n  requests_per_unit: 10000"]
    end

    subgraph Meaning["What each rule means"]
        M1["Marketing messages:\n5 per user per day\n(spam prevention)"]
        M2["Login endpoint:\n5 per second globally\n(brute-force protection)"]
        M3["Any request:\n10,000 per second\n(system-wide ceiling)"]
    end

    R1 --- M1
    R2 --- M2
    R3 --- M3

    style R1 fill:#2980B9,color:#fff,stroke:none
    style R2 fill:#27AE60,color:#fff,stroke:none
    style R3 fill:#8E44AD,color:#fff,stroke:none
    style M1 fill:#EBF5FB,color:#2980B9,stroke:#2980B9
    style M2 fill:#EAFAF1,color:#27AE60,stroke:#27AE60
    style M3 fill:#F5EEF8,color:#8E44AD,stroke:#8E44AD
"""))


### What this shows

Three rule nodes on the left (blue, green, purple) each showing a YAML-style config block with domain, descriptor key/value, and rate_limit parameters. Three meaning nodes on the right explain the plain-English interpretation of each rule. Horizontal dashed connectors link each config to its meaning. This maps the abstract config format to the real-world rate limiting scenario it addresses.

## 5. Handling Rate-Limited Requests: Drop vs Enqueue

> "In case a request is rate limited, APIs return HTTP response code 429 (too many requests). Depending on the use case, we can enqueue the rate-limited requests to be processed later." — Day 13

**Two strategies for handling excess requests:**

**Drop (immediate rejection):**
Return HTTP 429 immediately. The client reads `Retry-After` and retries after the window resets. Used for: login, payments, real-time feeds — operations that are time-sensitive and cannot be deferred.

**Enqueue (deferred processing):**
Place the excess request into a message queue. A worker picks it up when capacity is available. Used for: email delivery, report generation, batch jobs — operations where a slight delay is acceptable but the work must eventually complete.

**The three rate-limit response headers:**
- `X-RateLimit-Remaining` — requests left in current window
- `X-RateLimit-Limit` — total allowed per window
- `X-RateLimit-Retry-After` — seconds until retry is safe

In [5]:
# Cell 4 — Handling rate-limited requests: drop vs enqueue
display(mermaid("""
flowchart TD
    REQ([Incoming Request]) --> RL{Rate Limiter\nCheck}

    RL -->|"✅ Under limit"| API[API Server\nprocesses normally]
    API --> RES([HTTP 200 OK\nwith headers])

    RL -->|"❌ Over limit"| RESP([HTTP 429\nToo Many Requests])

    RESP --> HD["Response Headers\n─────────────────────────\nX-RateLimit-Remaining: 0\nX-RateLimit-Limit: 100\nX-RateLimit-Retry-After: 30"]

    RL -->|"❌ Over limit\n(non-critical path)"| DE{Drop or\nEnqueue?}

    DE -->|"Critical:\ndrop immediately"| DROP[Request Dropped\n→ client retries\nafter Retry-After]
    DE -->|"Deferrable:\nenqueue"| MQ[("Message Queue\n(process later)")]
    MQ --> W["Worker processes\nqueued requests\nwhen capacity frees"]
    W --> API2[API Server\n(async processing)]

    subgraph Examples["Use-case guide"]
        E1["Drop: login, payment,\nreal-time feeds"]
        E2["Enqueue: email delivery,\nreports, batch jobs"]
    end

    DROP -.-> E1
    MQ -.-> E2

    style REQ fill:#2E86AB,color:#fff,stroke:none
    style RL fill:#E67E22,color:#fff,stroke:none
    style API fill:#8E44AD,color:#fff,stroke:none
    style RES fill:#27AE60,color:#fff,stroke:none
    style RESP fill:#C44E52,color:#fff,stroke:none
    style HD fill:#FDECEA,color:#C44E52,stroke:#C44E52
    style DE fill:#F39C12,color:#fff,stroke:none
    style DROP fill:#C44E52,color:#fff,stroke:none
    style MQ fill:#2980B9,color:#fff,stroke:none
    style W fill:#2980B9,color:#fff,stroke:none
    style API2 fill:#8E44AD,color:#fff,stroke:none
"""))


### What this shows

A top-down flowchart from 'Incoming Request' through the rate limiter check. The allowed path (green) goes straight to API server → HTTP 200. The blocked path forks into two: HTTP 429 with the three response headers shown explicitly, and a separate 'Drop or Enqueue' decision diamond. The enqueue path shows Message Queue → Workers → API Server (async). A use-case guide subgraph labels which pattern fits each scenario.

## 6. Detailed Design Flow: Full Sequence

> "The rate limiter middleware loads rules from the cache. It fetches counters and last request timestamp from Redis cache. Based on the response, the rate limiter decides: if not rate limited, forward to API servers; if rate limited, return 429 and optionally enqueue." — Day 13

The sequence diagram below shows the exact message order between all five components for both the allowed and blocked paths, including the specific Redis commands used at each step.

In [6]:
# Cell 5 — Detailed design flow: rules cache + Redis counter + middleware decision
display(mermaid("""
sequenceDiagram
    participant CL as Client
    participant RL as Rate Limiter Middleware
    participant RC as Rules Cache (in-memory)
    participant RD as Redis Counter Store
    participant API as API Server

    CL->>RL: HTTP Request (user_id, endpoint)

    RL->>RC: 1 · Load rules for this endpoint
    RC-->>RL: rule: max=100 req/min

    RL->>RD: 2 · INCR rate:{user}:{window}
    RD-->>RL: counter = 47

    RL->>RD: 2b · EXPIRE rate:{user}:{window} 60
    RD-->>RL: TTL set ✓

    alt counter ≤ limit (47 ≤ 100)
        RL->>API: 3a · Forward request
        API-->>RL: HTTP 200 + body
        RL-->>CL: HTTP 200\nX-RateLimit-Remaining: 53\nX-RateLimit-Limit: 100
    else counter > limit
        RL-->>CL: HTTP 429 Too Many Requests\nX-RateLimit-Remaining: 0\nX-RateLimit-Retry-After: 30
        note over RL: optionally enqueue\nfor later processing
    end
"""))


### What this shows

A five-actor sequence diagram: Client, Rate Limiter Middleware, Rules Cache, Redis Counter Store, API Server.

**Message order:**
1. Client → Middleware: HTTP request with user_id and endpoint
2. Middleware → Rules Cache: load rules for this endpoint
3. Middleware → Redis: `INCR rate:{user}:{window}`
4. Middleware → Redis: `EXPIRE rate:{user}:{window} 60`
5. **Alt block (allowed):** forward to API Server → HTTP 200 with remaining header
6. **Alt block (blocked):** HTTP 429 with Retry-After header + optional enqueue

The `alt` block shows both paths in a single diagram, making the decision point explicit.

## 7. Race Condition & Synchronisation in Distributed RL

> "Scaling the system to support multiple servers and concurrent threads is a different story. There are two challenges: race condition and synchronisation issue." — Day 13

**Race condition:**
Two gateway nodes both read counter = 4 (limit = 5). Both decide to allow. Both write counter = 5. One extra request slips through — the limit is violated by 1.

**Fix:** Use a Redis Lua script — a server-side script that executes atomically. No other command can interleave between the read and the write:
```lua
local c = redis.call('INCR', key)
if c == 1 then redis.call('EXPIRE', key, ttl) end
return c
```

**Synchronisation issue:**
If each gateway node keeps its own local counter (no shared state), a user routed across nodes can exceed the limit without any single node knowing.

**Fix:** Centralised Redis cluster as the single shared counter store — all gateway nodes read and write the same key.

In [7]:
# Cell 6 — Race condition and synchronisation challenges in distributed RL
display(mermaid("""
flowchart TD
    subgraph Problem1["Race Condition (2 nodes, 1 Redis key)"]
        direction LR
        N1["Node 1\nreads counter = 4"]
        N2["Node 2\nreads counter = 4"]
        N1 -->|"both see 4 < 5\nboth write 5"| RK[("Redis key = 5\n(should be 6!)")]
        N2 --> RK
        RK --> OVER["❌ One extra request\nslips through limit"]
    end

    subgraph Fix1["Fix: Lua Script (atomic read-write)"]
        direction LR
        LUA["Redis Lua Script\n──────────────────\nlocal c = redis.call('INCR', key)\nif c == 1 then\n  redis.call('EXPIRE', key, ttl)\nend\nreturn c"]
        LUA --> AT["✅ Atomic — no\ninterleaving possible"]
    end

    subgraph Problem2["Synchronisation (sticky session naive fix)"]
        direction TB
        LB2[Load Balancer]
        GW1["Gateway Node 1\nlocal store: user42=3"]
        GW2["Gateway Node 2\nlocal store: user42=3"]
        LB2 --> GW1
        LB2 --> GW2
        GW1 -.->|"no sync"| GW2
        GW1 --> WRONG["❌ Same problem:\n6 req served at limit 5"]
    end

    subgraph Fix2["Fix: Centralised Redis (shared state)"]
        direction TB
        LB3[Load Balancer]
        GW3["Gateway Node 1"]
        GW4["Gateway Node 2"]
        RS[("Redis Cluster\nShared Counter")]
        LB3 --> GW3
        LB3 --> GW4
        GW3 -->|"INCR"| RS
        GW4 -->|"INCR"| RS
        RS --> OK2["✅ All nodes see\nsame counter"]
    end

    Problem1 --> Fix1
    Problem2 --> Fix2

    style OVER fill:#C44E52,color:#fff,stroke:none
    style AT fill:#27AE60,color:#fff,stroke:none
    style WRONG fill:#C44E52,color:#fff,stroke:none
    style OK2 fill:#27AE60,color:#fff,stroke:none
    style LUA fill:#2980B9,color:#fff,stroke:none
    style RS fill:#C44E52,color:#fff,stroke:none
"""))


### What this shows

Four subgraphs arranged in two problem/fix pairs:

**Problem 1 (Race Condition):** Two nodes both read `counter = 4`, both allow, both write `5` to the same Redis key — one extra request passes. The result node is red with '❌ One extra request slips through'.

**Fix 1 (Lua Script):** A Lua script box shows the atomic INCR + EXPIRE code. Result: green '✅ Atomic — no interleaving possible'.

**Problem 2 (Synchronisation):** A load balancer splits traffic to two gateway nodes with independent local stores. No sync arrow between them. Same outcome — 6 requests served at limit 5.

**Fix 2 (Centralised Redis):** Both gateways INCR the same Redis Cluster key. Green '✅ All nodes see same counter'.

## 8. Complete Rate Limiter Architecture

In [8]:
# Cell 7 — Complete architecture summary + evolution Day 1–13
display(mermaid("""
flowchart TD
    subgraph External["External Traffic"]
        CL([Client / Browser / App])
    end

    subgraph Gateway["API Gateway Layer"]
        GW["API Gateway\n+ Rate Limiter Middleware"]
        RC2[("Rules Cache\n(fast in-memory)")]
        GW <-->|"load rules"| RC2
    end

    subgraph Counter["Counter Store"]
        RD2[("Redis Cluster\nINCR · EXPIRE · Lua")]
        GW <-->|"atomic\ncounter ops"| RD2
    end

    subgraph RulesMgmt["Rules Management"]
        DISK[("Rules on Disk\n(YAML / config)")]
        WKR["Worker\n(periodic pull)"]
        DISK -->|"fetch"| WKR
        WKR -->|"populate"| RC2
    end

    subgraph Backend["Backend Services"]
        API1["Service A"]
        API2["Service B"]
        API3["Service C"]
    end

    subgraph Overflow["Rate-Limited Handling"]
        MQ2[("Message Queue\n(deferred work)")]
        WRKR2["Async Workers"]
        MQ2 --> WRKR2
        WRKR2 --> API1
    end

    CL -->|"HTTP request"| GW
    GW -->|"✅ under limit\nforward"| API1
    GW -->|"✅ under limit\nforward"| API2
    GW -->|"✅ under limit\nforward"| API3
    GW -->|"❌ over limit\nHTTP 429 + headers"| CL
    GW -->|"❌ deferrable\nenqueue"| MQ2

    style CL fill:#2E86AB,color:#fff,stroke:none
    style GW fill:#E67E22,color:#fff,stroke:none
    style RC2 fill:#2980B9,color:#fff,stroke:none
    style RD2 fill:#C44E52,color:#fff,stroke:none
    style DISK fill:#8E44AD,color:#fff,stroke:none
    style WKR fill:#8E44AD,color:#fff,stroke:none
    style API1 fill:#27AE60,color:#fff,stroke:none
    style API2 fill:#27AE60,color:#fff,stroke:none
    style API3 fill:#27AE60,color:#fff,stroke:none
    style MQ2 fill:#2980B9,color:#fff,stroke:none
    style WRKR2 fill:#2980B9,color:#fff,stroke:none
"""))


### What this shows

The full production architecture in a single diagram, combining every component from Parts 1, 2, and 3:

**External Traffic:** Client sends HTTP requests.

**API Gateway Layer:** Rate Limiter Middleware intercepts all requests. It consults the Rules Cache (fast in-memory store) to load the applicable rule.

**Counter Store:** Redis Cluster handles atomic INCR/EXPIRE and Lua scripts. All gateway nodes share this single source of truth.

**Rules Management:** Rules on Disk → Worker (periodic pull) → Rules Cache. Rules update without touching the counter store or restarting the gateway.

**Backend Services:** Three API services receive only allowed traffic.

**Rate-Limited Handling:** Blocked requests either return HTTP 429 directly, or are placed into a Message Queue for async worker processing.

**Key takeaway:** Every component has a single responsibility. The rate limiter sits between the client and the backend — it is transparent to backend services and invisible to compliant clients.

## AWS Production Notes

> Based on the AWS Well-Architected Framework and AWS official documentation.

---

### Critical Corrections

| Claim in This Notebook | Correction |
|---|---|
| 'Two gateway nodes both read counter = 4. Both decide to allow. Both write counter = 5. One extra request slips through.' | **Redis INCR is atomic.** Two concurrent INCR calls return 5 and 6 respectively — they cannot both return 5. The actual race condition is **INCR and EXPIRE being two separate commands**: a crash after INCR but before EXPIRE creates a persistent key with no TTL, permanently blocking the user. |
| Sequence diagram: EXPIRE called on every request after every INCR | **Calling EXPIRE on every request resets the TTL window with each new request** — a user making steady requests never triggers the window reset, breaking the rate limit. EXPIRE must only be set when the key is first created (counter == 1). The Lua script in Section 7 is correct; the sequence diagram contradicts it. |
| Header listed as 'X-RateLimit-Retry-After' | **`X-RateLimit-Retry-After` is not a real header.** The correct header is `Retry-After` (RFC 6585 Section 4). Emerging IETF standard defines: `RateLimit-Limit`, `RateLimit-Remaining`, `RateLimit-Reset` (no `X-` prefix). |
| 'Rate limiting rules saved on disk; workers pull from disk' | In AWS, flat files on disk are not a production pattern — EC2/container filesystems are ephemeral and non-centralized. **AWS AppConfig** (versioned, validated, deployment strategies, auto-rollback) or **SSM Parameter Store** (versioned, encrypted) are the correct AWS patterns. |

---

### Corrected Lua Script (Production-Safe)

```lua
-- Atomic INCR + conditional EXPIRE
-- Runs on Redis server — no interleaving possible
local key    = KEYS[1]
local limit   = tonumber(ARGV[1])
local window  = tonumber(ARGV[2])

local count = redis.call('INCR', key)
if count == 1 then
    -- EXPIRE set ONLY on first creation — window resets naturally
    -- If set on every request: window never resets for active users
    redis.call('EXPIRE', key, window)
end
return count
-- Returns: counter value. Caller checks if count > limit.
```

---

### Replace 'Rules on Disk' with AWS AppConfig

```
Rules on disk (anti-pattern on AWS):
  - EC2/container filesystem is ephemeral
  - Not centralized — each instance reads its own file
  - No versioning, validation, or rollback

AWS AppConfig (production pattern):
  - Versioned configurations with deployment strategies
  - Canary/linear rollout: deploy to 10% of instances first
  - Automatic rollback if CloudWatch alarm triggers
  - All instances read from same source
  - SDK: appconfig.get_latest_configuration() — caches locally, polls for changes

AWS SSM Parameter Store (simpler alternative):
  - Key/value store for config and secrets
  - Versioned; encrypted with KMS
  - IAM-controlled access
  - Free for Standard parameters
```

---

### ElastiCache Global Datastore: Cross-Region Rate Limiting

```
Problem: Rate limit counters stored in us-east-1 are invisible to eu-west-1.
         User makes 5 requests to each region → 10 requests total bypass a limit of 5.

Solution: ElastiCache Global Datastore
  - Active-active replication across up to 5 regions
  - < 1s typical replication lag
  - All regions write to local primary; replication is async
  - Provides eventual consistency (not strong consistency) for rate limit counters

Acceptable for most rate limiting use cases — the goal is to prevent abuse,
not to enforce microsecond-exact counts.
```

---

### Updated Complete Architecture with AWS Services

```
Client
  ↓ HTTPS
AWS Shield Standard (L3/L4 DDoS — automatic, free)
  ↓
Amazon CloudFront
  ↓
AWS WAF  (rate-based rules: IP-level, edge, no code)
  ↓
Amazon API Gateway (Usage Plans: per-client token bucket)
  ↓
ALB → EC2 Rate Limiter Middleware
         ├── AWS AppConfig / SSM ← rate limit rules (versioned, validated)
         └── ElastiCache (Valkey/Redis, Multi-AZ)
                   ↑ Lua INCR script (atomic)
                   ├── ✅ ALLOW → downstream API service
                   └── ❌ BLOCK → HTTP 429 with Retry-After header
                              OR → Amazon SQS (deferred work queue)
```

---

### Key CloudWatch Metrics for Rate Limiting

| Metric | Service | Alarm Condition |
|---|---|---|
| `ThrottleCount` | API Gateway | > 100 over 5 min |
| `BlockedRequests` | AWS WAF | > 500 over 5 min (investigate source) |
| `CacheHits` / `CacheMisses` | ElastiCache | Hit rate drop → rate limiter slowdown |
| `ReplicationLag` | ElastiCache Global Datastore | > 5s means cross-region counters are stale |
| `DeploymentStatus` | AppConfig | Failed deployment → rules didn't update |

---

### Well-Architected Checklist

- [ ] **SEC 6** — WAF rate-based rules at CloudFront/ALB edge; Shield Standard enabled
- [ ] **SEC 2** — Rate limit key = authenticated identity (Cognito sub, IAM ARN, API Key) — not IP
- [ ] **REL 10** — Fail-open with local fallback counter (not full fail-open) when ElastiCache is unavailable
- [ ] **REL 9** — ElastiCache Multi-AZ (not Redis Sentinel); ~30–60s automatic failover
- [ ] **OPS 5** — Rules in AWS AppConfig (not files on disk); versioned, validated, auto-rollback
- [ ] **OPS 6** — CloudWatch alarms on `ThrottleCount`, `BlockedRequests`, ElastiCache `CurrConnections`
- [ ] **PERF 1** — API Gateway Usage Plans first; custom ElastiCache middleware only for business-logic limits
